# GitHub PR Review

Thin notebook wrapper for the PR-review LangGraph implementation in `github_review_util.py`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

import gradio as gr

util_candidates = [
    Path.cwd(),
    Path.cwd() / "4_langgraph" / "community_contributions" / "misi",
]

for candidate in util_candidates:
    if (candidate / "github_review_util.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from github_review_util import (
    build_review_graph,
    default_model,
    default_repository,
    default_supervisor_model,
    load_review_environment,
    new_thread_config,
)

load_review_environment()
review_graph = build_review_graph()
supervised_graph = build_review_graph(use_supervisor=True)

print(f"Default repository: {default_repository() or '(not set)'}")
print(f"Review model: {default_model()}")
print(f"Supervisor model: {default_supervisor_model()}")

In [ ]:
print("\nReview graph:")
display(Image(review_graph.get_graph().draw_mermaid_png()))
print("\nSupervised review graph:")
display(Image(supervised_graph.get_graph().draw_mermaid_png()))

In [ ]:
def gradio_messages(user_input, history):
    messages = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in {"user", "assistant"} and content:
                messages.append({"role": role, "content": content})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item
            if user_message:
                messages.append({"role": "user", "content": user_message})
            if assistant_message:
                messages.append({"role": "assistant", "content": assistant_message})
    messages.append({"role": "user", "content": user_input})
    return messages


async def chat(user_input, history, use_supervisor):
    active_graph = supervised_graph if use_supervisor else review_graph
    result = await active_graph.ainvoke(
        {"messages": gradio_messages(user_input, history)},
        config=new_thread_config(),
    )
    return result.get("review") or result["messages"][-1].content


with gr.Blocks() as demo:
    gr.Markdown("# GitHub PR Reviewer")
    use_supervisor = gr.Checkbox(label="Use supervisor", value=False)
    gr.ChatInterface(
        chat,
        type="messages",
        additional_inputs=[use_supervisor],
    )

demo.launch()